# Capstone — Wanderbricks Platform Intelligence

**Dataset:** `samples.wanderbricks` — bookings, users, properties, reviews, payments, hosts

**Difficulty:** Hard

**Topics:** multi-table joins, window functions, datediff, collect_set, conditional columns, percentage calculations

> These problems are **linked** — each one adds depth to the one before it.
> Work through them in order: the join patterns and business context carry forward.
>
> Business context: you are a data analyst at Wanderbricks, an Airbnb-style rental platform.
> Leadership wants a series of reports covering host performance, guest behaviour, booking
> patterns, and platform health.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings   = spark.table("samples.wanderbricks.bookings")
users      = spark.table("samples.wanderbricks.users")
properties = spark.table("samples.wanderbricks.properties")
reviews    = spark.table("samples.wanderbricks.reviews")
payments   = spark.table("samples.wanderbricks.payments")
hosts      = spark.table("samples.wanderbricks.hosts")

---
## Problem 1 — Active Host Portfolio

The platform team needs a ranked view of hosts by the size of their active listing portfolio.

Join `properties` → `hosts` (where `hosts.is_active = true`).

Per host, compute:
- `property_count` — number of active properties
- `avg_base_price` — average `base_price`, rounded to 2 decimal places
- `property_types` — `collect_set` of distinct property types they list

Return the **top 20 hosts** by `property_count` descending.

**Expected output columns:** `host_id`, `name`, `country`, `is_verified`, `property_count`, `avg_base_price`, `property_types`

In [ ]:
# Problem 1 - write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# Tests for Problem 1
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'host_id' in cols, "Missing column: host_id"
assert 'property_count' in cols, "Missing column: property_count"
assert 'property_types' in cols, "Missing column: property_types"
cnt = result_1.count()
assert 0 < cnt <= 20, f"Expected 1-20 rows, got {cnt}"
from pyspark.sql.types import ArrayType
schema_dict = {f.name: f.dataType for f in result_1.schema.fields}
pt_col = [c for c in result_1.columns if 'type' in c.lower()][0]
assert isinstance(schema_dict[pt_col], ArrayType), f"{pt_col} must be ArrayType (use collect_set)"
print(f"Problem 1 passed ({cnt} hosts)")

---
## Problem 2 — Booking Revenue by Property Type and Country

Building on Problem 1's understanding of the property portfolio, now bring in booking revenue.

Join `bookings` (where `status = 'confirmed'`) → `properties` → `hosts`.

> **Note:** `country` is a column on `hosts`, not `properties` — you need all three tables.

Per `(property_type, country)` (host's country from the `hosts` table), compute:
- `total_bookings` — count of confirmed bookings
- `total_revenue` — sum of `total_amount`, rounded to 2 dp
- `avg_booking_value` — average `total_amount`, rounded to 2 dp
- `avg_stay_nights` — average `datediff(check_out, check_in)`, rounded to 1 dp

Sort by `total_revenue` descending.

**Expected output columns:** `property_type`, `country`, `total_bookings`, `total_revenue`, `avg_booking_value`, `avg_stay_nights`

In [ ]:
# Problem 2 - write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# Tests for Problem 2
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'property_type' in cols, "Missing column: property_type"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'avg_stay_nights' in cols, "Missing column: avg_stay_nights"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
revenues = [r['total_revenue'] for r in result_2.collect()]
assert revenues == sorted(revenues, reverse=True), "Must be sorted by total_revenue descending"
print(f"Problem 2 passed ({cnt} property_type/country combinations)")

---
## Problem 3 — Guest Spending Behaviour

The growth team wants to identify the platform's highest-value guests.

Join `bookings` (where `status = 'confirmed'`) → `users` (on `user_id`) → `properties` → `hosts` (to get host `country` for each property).

Per guest (`user_id`), compute:
- `trips_taken` — number of confirmed bookings
- `total_spend` — sum of `total_amount`, rounded to 2 dp
- `avg_stay_nights` — average `datediff(check_out, check_in)`, rounded to 1 dp
- `countries_visited` — `collect_set` of distinct host countries across all bookings

Return the **top 20 guests** by `total_spend` descending.

**Expected output columns:** `user_id`, `trips_taken`, `total_spend`, `avg_stay_nights`, `countries_visited`

In [ ]:
# Problem 3 - write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# Tests for Problem 3
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'user_id' in cols, "Missing column: user_id"
assert 'total_spend' in cols, "Missing column: total_spend"
assert 'countries_visited' in cols, "Missing column: countries_visited"
cnt = result_3.count()
assert 0 < cnt <= 20, f"Expected 1-20 rows, got {cnt}"
from pyspark.sql.types import ArrayType
schema_dict = {f.name: f.dataType for f in result_3.schema.fields}
cv_col = [c for c in result_3.columns if 'countr' in c.lower()][0]
assert isinstance(schema_dict[cv_col], ArrayType), f"{cv_col} must be ArrayType (use collect_set)"
print(f"Problem 3 passed ({cnt} guests)")

---
## Problem 4 — Review Quality by Property Type

Building on Problem 2's property type breakdown, now bring in guest reviews.

Join `reviews` (where `is_deleted = false`) → `properties`.

Per `property_type`, compute:
- `avg_rating` — average rating, rounded to 2 dp
- `review_count` — total number of non-deleted reviews
- `rated_property_count` — count of distinct `property_id`s that received reviews
- `quality_flag` — `'Needs Attention'` if `avg_rating < 3.5`, otherwise `'Good'`

Sort by `avg_rating` descending.

**Expected output columns:** `property_type`, `avg_rating`, `review_count`, `rated_property_count`, `quality_flag`

In [ ]:
# Problem 4 - write your solution here
# Assign result to: result_4

result_4 = None  # replace this

In [ ]:
# Tests for Problem 4
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'property_type' in cols, "Missing column: property_type"
assert 'avg_rating' in cols, "Missing column: avg_rating"
assert 'quality_flag' in cols, "Missing column: quality_flag"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
flags = {r['quality_flag'] for r in result_4.select('quality_flag').collect()}
assert flags.issubset({'Good', 'Needs Attention'}), f"Unexpected quality_flag values: {flags}"
print(f"Problem 4 passed ({cnt} property types, flags: {flags})")

---
## Problem 5 — Cancellation Intelligence

The operations team needs to understand where cancellations cluster.

Join `bookings` → `users` (on `user_id`) → `properties`.

Per `(property_type, country)` (user country from `users`), compute:
- `total_bookings` — all bookings
- `cancelled_bookings` — bookings where `status = 'cancelled'`
- `cancellation_rate_pct` — `cancelled_bookings / total_bookings * 100`, rounded to 2 dp

Only include combinations with at least 5 total bookings.
Sort by `cancellation_rate_pct` descending.

**Expected output columns:** `property_type`, `country`, `total_bookings`, `cancelled_bookings`, `cancellation_rate_pct`

In [ ]:
# Problem 5 - write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# Tests for Problem 5
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'cancellation_rate_pct' in cols, "Missing column: cancellation_rate_pct"
assert 'total_bookings' in cols, "Missing column: total_bookings"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
min_bookings = result_5.agg(F.min('total_bookings')).collect()[0][0]
assert min_bookings >= 5, f"All groups must have >= 5 total bookings, min was {min_bookings}"
max_pct = result_5.agg(F.max('cancellation_rate_pct')).collect()[0][0]
assert max_pct <= 100, f"cancellation_rate_pct must be <= 100, got {max_pct}"
print(f"Problem 5 passed ({cnt} combinations)")

---
## Problem 6 — Seasonal Booking Patterns

The product team wants to understand which months drive the most bookings per property type.

From `bookings` (all statuses) → `properties`:

1. Group by `(property_type, check_in month)` and compute `booking_count` and `monthly_revenue` (sum of `total_amount`)
2. Use a **window function** to rank months within each property type by `booking_count` (rank 1 = busiest)
3. Keep only the **top 3 months** per property type (rank <= 3)
4. Use `F.when()` to map month numbers to names (`1` → `'January'`, `2` → `'February'`, etc.)
5. Use `collect_list` to produce one row per property type with an array of peak month names

**Expected output columns:** `property_type`, `peak_months` (array of month name strings), `total_yearly_bookings`

In [ ]:
# Problem 6 - write your solution here
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# Tests for Problem 6
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'property_type' in cols, "Missing column: property_type"
assert 'peak_months' in cols, "Missing column: peak_months"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
from pyspark.sql.types import ArrayType
schema_dict = {f.name: f.dataType for f in result_6.schema.fields}
pm_col = [c for c in result_6.columns if 'month' in c.lower()][0]
assert isinstance(schema_dict[pm_col], ArrayType), f"{pm_col} must be ArrayType (use collect_list)"
valid_months = {'January','February','March','April','May','June',
                'July','August','September','October','November','December'}
for row in result_6.select(pm_col).collect():
    for month in row[0]:
        assert month in valid_months, f"Unexpected month name: {month}"
print(f"Problem 6 passed ({cnt} property types)")

---
## Problem 7 — Host Performance Dashboard

The final report combines everything: host profile, bookings, completed payments, and reviews.

Join: `properties` → `hosts` + `bookings` → `payments` (where `status = 'completed'`) + `reviews` (where `is_deleted = false`).

Per host, compute:
- `total_properties` — distinct count of properties they own
- `total_confirmed_bookings` — bookings where `status = 'confirmed'`
- `total_revenue` — sum of completed `payments.amount`, rounded to 2 dp
- `avg_guest_rating` — average review rating across all their properties, rounded to 2 dp
- `cancellation_rate_pct` — `cancelled bookings / total bookings * 100`, rounded to 2 dp

Only include hosts with at least 1 completed payment.
Sort by `total_revenue` descending. Limit to 25 rows.

**Expected output columns:** `host_id`, `name`, `country`, `is_verified`, `total_properties`, `total_confirmed_bookings`, `total_revenue`, `avg_guest_rating`, `cancellation_rate_pct`

In [ ]:
# Problem 7 - write your solution here
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# Tests for Problem 7
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'host_id' in cols, "Missing column: host_id"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'avg_guest_rating' in cols, "Missing column: avg_guest_rating"
assert 'cancellation_rate_pct' in cols, "Missing column: cancellation_rate_pct"
cnt = result_7.count()
assert 0 < cnt <= 25, f"Expected 1-25 rows, got {cnt}"
revenues = [r['total_revenue'] for r in result_7.collect()]
assert revenues == sorted(revenues, reverse=True), "Must be sorted by total_revenue descending"
max_cancel = result_7.agg(F.max('cancellation_rate_pct')).collect()[0][0]
assert max_cancel <= 100, f"cancellation_rate_pct must be <= 100, got {max_cancel}"
print(f"Problem 7 passed ({cnt} hosts)")